In [ ]:
# Phase 1: Visual Studio Project Setup
#Install Essential Libraries
%pip install transformers datasets torch accelerate scikit-learn

<!-- Phase 2: The Code (The "Scratch" Build) -->

In [ ]:
##Load the Dataset
#We’ll use the IMDb movie review dataset from the Hugging Face Hub.
from datasets import load_dataset

# Load a small portion for speed (optional)
dataset = load_dataset("imdb")
print(dataset["train"][0]) # View one example

In [ ]:
#Preprocess (Tokenization)
#We must convert text into numbers that DistilBERT understands using the DistilBertTokenizer.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply tokenization to the whole dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)


In [ ]:
#3. Load the Model
#We load the "Base" DistilBERT with a classification head attached to the top.
from transformers import AutoModelForSequenceClassification

# num_labels=2 for Positive and Negative
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


In [ ]:
# 4. Define Metrics
# The model outputs raw numbers; we need to calculate Accuracy.
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


In [ ]:
# 5. The Training Loop (Trainer API)
# This is where the magic happens. We define the hyperparameters and start training.
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="test_trainer",
    evaluation_strategy="epoch",  # Check accuracy after every epoch
    per_device_train_batch_size=8, # Adjust based on your GPU memory
    num_train_epochs=1,            # Just 1 for the "Hello World" test
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(1000)), # Sample for speed
    eval_dataset=tokenized_datasets["test"].shuffle(seed=42).select(range(1000)),
    compute_metrics=compute_metrics,
)

trainer.train()


<!-- Phase 3: Testing Your Model
Once training is done, test it on a manual sentence: -->

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
print(classifier("I absolutely loved this movie! The acting was phenomenal."))
